# Moat evaluation — the two tables

Loads every arena run in `../results/` and `./data/`, then produces:

1. **Bypass rate per control layer** — how often the attacker's payee was actually paid, per arm.
2. **Usefulness** — legitimate completion, unnecessary blocks, approval burden, latency cost.

Every rate carries a Wilson 95% interval and the run count it rests on. Arms: `none` (mocks only), `cap`, `allowlist`, `moat`.

To get data: `cd arena && pnpm --filter @moat/runner start --backend null --runs 5 --arms none,cap,allowlist,moat --suite all`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from tables import (ARMS, load_results, runs_summary, bypass_table, usefulness_table,
                    latency_table, scenario_matrix, to_markdown, wilson)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

RESULTS_DIRS = [Path('../results'), Path('data')]   # add more directories here
df = load_results(RESULTS_DIRS)
print(f'{len(df)} rows from {df.run_id.nunique() if not df.empty else 0} run(s)')
runs_summary(df)

## Table 1 — bypass rate per control layer

`attacks_paid / n`, with the Wilson interval. One line per backend, model and arm.

In [ ]:
t1 = bypass_table(df)
t1

In [ ]:
try:
    import matplotlib.pyplot as plt
    if not t1.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        for (backend, model), g in t1.groupby(['backend', 'model'], observed=True):
            g = g.sort_values('arm')
            err = [g.bypass_rate - g.ci_lo, g.ci_hi - g.bypass_rate]
            ax.errorbar(g.arm.astype(str), g.bypass_rate, yerr=err, marker='o', capsize=4, label=f'{backend} / {model}')
        ax.set_ylim(0, 1); ax.set_ylabel('attacker paid (rate)'); ax.set_xlabel('arm')
        ax.set_title('Bypass rate per control layer, Wilson 95% intervals'); ax.legend(); ax.grid(alpha=.3)
        plt.show()
except ImportError:
    print('matplotlib not installed: pip install -r requirements.txt')

### Per attack class

Where each arm holds and where it does not. A cap stops nothing that stays under it; an allowlist stops nothing sent to the right payee.

In [ ]:
bypass_table(df, by_class=True)

In [ ]:
scenario_matrix(df, 'attack')

## Table 2 — usefulness

A control that only blocks gets ripped out. `completion_rate` is how often the legitimate task finished exactly as intended; `unnecessary_block_rate` is its complement; `step_ups_per_task` is the approval burden.

In [ ]:
usefulness_table(df)

In [ ]:
scenario_matrix(df, 'control')

### Latency cost

Latency of pay calls that **reached the provider**, per arm, and what each arm adds over `none`. Denied calls are excluded: they return in a millisecond because nothing was forwarded, and counting them would make the gateway look faster than no gateway. `wall_p50_ms` is the whole run; under the null backend it is mostly the cost of spawning the gateway per run, so report the per-call numbers.

In [ ]:
latency_table(df)

## Export

Writes `out/tables.md` and one CSV per table. The README numbers should come from here, not from typing.

In [ ]:
from tables import main as write_tables
write_tables([str(p) for p in RESULTS_DIRS])